In [ ]:
# from rdkit.Chem import PandasTools
import numpy as np
import pandas as pd
# from rdkit import DataStructs
# from rdkit.Chem import AllChem as Chem
# from rdkit.Chem import Draw
# from rdkit.Chem import Descriptors
# from rdkit.ML.Descriptors import MoleculeDescriptors
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
import random
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.model_selection import LeaveOneOut
from sklearn import preprocessing
#from genetic_selection import GeneticSelectionCV
# from mordred import Calculator, descriptors

In [ ]:
dataA = pd.read_csv("../data/MLdataA.csv")
dataB = pd.read_csv("../data/MLdataB.csv")
data = pd.read_csv("../data/allData.csv")


In [ ]:
# dfsmile[dfsmile.columns[1:4]]

In [ ]:
# dfsmile["inp+inp=out"] = dfsmile["input"] + "="+dfsmile["output"] 

In [ ]:
# dfsmile

In [ ]:
# dt = pd.concat([dfsmile["inp+inp=out"], data[data.columns[1:]]], axis=1)

In [ ]:
# dt

In [ ]:
data

In [ ]:
data["inp+inp=out"][4]

In [ ]:
# dt.to_csv('../data/allData.csv', index=False)

## Filter reactions
Drop rows where target **b** is zero and reset the index.


In [ ]:
df_no_zeroB = data[data['b'] != 0]
df_no_zeroB = df_no_zeroB.reset_index()

In [ ]:
# X = data.iloc[:, 3:]

## Genetic feature selection
Run `GAFeatureSelectionCV` with Extra Trees to shrink the descriptor space for **a** and **b**.


In [ ]:
from sklearn_genetic import GAFeatureSelectionCV
import inspect
def extra_trees_genetic_selection(X, y, n_features=10):
    """
    Genetic feature selection with Extra Trees
    """
    print("=== EXTRA TREES GENETIC FEATURE SELECTION ===")
    print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
    
    et = ExtraTreesRegressor(
        n_estimators=100,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.2,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )
    
    model = GAFeatureSelectionCV(
        estimator=et,
        cv=3,
        scoring="neg_mean_squared_error",
        population_size=50,
        generations=25,
        max_features=n_features,
        crossover_probability=0.75,
        mutation_probability=0.25,
        tournament_size=3,
        elitism=True,
        verbose=True,
        n_jobs=1,
        error_score='raise' 
    )

    print("\nStarting Extra Trees genetic selection...")
    model.fit(X, y)
    
    selected_features = X.columns[model.support_].tolist()
    
    print(f"\n✓ Selection completed!")
    print(f"Selected {len(selected_features)} features:")
    for i, feat in enumerate(selected_features, 1):
        print(f"  {i:2d}. {feat}")
    
    return model, selected_features

In [ ]:
print("GAFeatureSelectionCV parameters:")
sig = inspect.signature(GAFeatureSelectionCV.__init__)
for param_name, param in sig.parameters.items():
    if param_name != 'self':
        print(f"  - {param_name}")

In [ ]:
data

In [ ]:
data = data.loc[:, (data != 0).any(axis=0)]


In [ ]:
data

In [ ]:
model_a, features_a = extra_trees_genetic_selection(data.iloc[:, 3:], data["a"], n_features=35)


In [ ]:
df_no_zeroB = df_no_zeroB.loc[:, (df_no_zeroB != 0).any(axis=0)]


In [ ]:
df_no_zeroB= df_no_zeroB[df_no_zeroB.columns[1:]]

In [ ]:
df_no_zeroB

In [ ]:
model_b, features_b = extra_trees_genetic_selection(df_no_zeroB.iloc[:, 3:], df_no_zeroB["b"], n_features=35)


## Persist or reload artefacts
Pickle / joblib utilities for feature lists and models.


In [ ]:
import pickle

In [ ]:
# with open('features_a.pkl', 'wb') as file:
#     pickle.dump(features_a, file)
# with open('features_b.pkl', 'wb') as file:
#     pickle.dump(features_b, file)

In [ ]:
import os
os.getcwd()

In [ ]:
with open('../code/streamlit/features_a2.pkl', 'rb') as file:
    features_a = pickle.load(file)

In [ ]:
with open('../code/streamlit/features_b2.pkl', 'rb') as file:
    features_b = pickle.load(file)

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## Helper utilities
Search helpers and reusable analysis functions.


## Extra Trees leave-one-out driver
Custom LOO-style workflow with optional diagnostic plots.


In [ ]:
def max_search(data, features, name):
    def extra_trees_loo_analysis12(X, y, random_state=24):
        """
        Полный анализ Extra Trees с Leave-One-Out и предсказаниями на новых данных
        """
        # X_train, X_new, y_train, y_new = train_test_split(
        #     X, y, test_size=test_size, random_state=random_state, shuffle=True
        # )
        
        # et_model = ExtraTreesRegressor(
        #     n_estimators=200,
        #     max_depth=4,
        #     min_samples_split=4,
        #     min_samples_leaf=3,
        #     max_features=0.6,
        #     bootstrap=True,
        #     random_state=42,
        #     n_jobs=-1
        # )
        et_model = ExtraTreesRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features=0.6,
            bootstrap=False,
            random_state=42,
            n_jobs=-1
        )
        
        
        loo = LeaveOneOut()
        loo_scores = []
        loo_predictions = []
        loo_true_values = []
        
        for train_idx, test_idx in loo.split(X):
            X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
            y_train_fold, y_test_fold = y.iloc[train_idx], y.iloc[test_idx]
            
            et_model.fit(X_train_fold, y_train_fold)
            y_pred_fold = et_model.predict(X_test_fold)
            
            mse_fold = mean_squared_error(y_test_fold, y_pred_fold)
            loo_scores.append(-mse_fold)
            loo_predictions.extend(y_pred_fold)
            loo_true_values.extend(y_test_fold)
        
        et_model.fit(X, y)
        
        y_train_pred = et_model.predict(X)
        
        train_r2 = r2_score(y, y_train_pred)
        train_mse = mean_squared_error(y, y_train_pred)
        train_mae = mean_absolute_error(y, y_train_pred)
        
        loo_r2 = r2_score(loo_true_values, loo_predictions)
        loo_mse = mean_squared_error(loo_true_values, loo_predictions)
        loo_mae = mean_absolute_error(loo_true_values, loo_predictions)

        return {
            'model': et_model,
            'X_train': X,
            'y_train': y,
            # 'X_new': X_new,
            # 'y_new': y_new,
            # 'y_new_pred': y_new_pred,
            'loo_scores': loo_scores,
            'loo_predictions': loo_predictions,
            'loo_true_values': loo_true_values,
            'metrics': {
                'train': {'r2': train_r2, 'mse': train_mse, 'mae': train_mae},
                'loo': {'r2': loo_r2, 'mse': loo_mse, 'mae': loo_mae},
            }
        }
    list = []
    ind = []
    for i in range(10, len(features)):
        results_a = extra_trees_loo_analysis12(data[features_a[:i]], data[name], random_state=42)
        list.append(results_a["metrics"]["loo"]["r2"])
        ind.append(i)
    fig, axes = plt.subplots(1, figsize=(15, 12))

    axes.plot(ind, list, alpha=0.7, color='skyblue')
    axes.set_title('Performance vs number of descriptors')
    axes.set_xlabel('Namber')
    axes.set_ylabel('R2')
    axes.grid(axis='x', alpha=0.3)
    return max(list)

In [ ]:
# max_search(df_no_zeroB, features_b, "b")

In [ ]:
# max_search(data, features_a, "a")

## Run LOO experiments
Execute the custom Extra Trees analysis and inspect hold-out behaviour.


In [ ]:
def extra_trees_loo_analysis_castom(X, y, test_size=5, random_state=42, picture = False):
    """
    Полный анализ Extra Trees с Leave-One-Out и предсказаниями на новых данных
    """
    print("=" * 80)
    print("FULL EXTRA TREES ANALYSIS WITH LEAVE-ONE-OUT")
    print("=" * 80)
    
    X_train, X_new, y_train, y_new = train_test_split(
        X, y, test_size=test_size, random_state=random_state, shuffle=False
    )
    
    print(f"Размер тренировочных данных: {X_train.shape}")
    print(f"Размер новых данных (модель никогда не видела): {X_new.shape}")

    et_model = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.6,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )
    
    print("\n" + "=" * 50)
    print("LEAVE-ONE-OUT CROSS-VALIDATION")
    print("=" * 50)
    
    loo = LeaveOneOut()
    loo_scores = []
    loo_predictions = []
    loo_true_values = []
    
    for train_idx, test_idx in loo.split(X_train):
        X_train_fold, X_test_fold = X_train.iloc[train_idx], X_train.iloc[test_idx]
        y_train_fold, y_test_fold = y_train.iloc[train_idx], y_train.iloc[test_idx]
        
        et_model.fit(X_train_fold, y_train_fold)
        y_pred_fold = et_model.predict(X_test_fold)
        
        mse_fold = mean_squared_error(y_test_fold, y_pred_fold)
        loo_scores.append(-mse_fold)
        loo_predictions.extend(y_pred_fold)
        loo_true_values.extend(y_test_fold)
    
    print("\n" + "=" * 50)
    print("FINAL MODEL FIT")
    print("=" * 50)
    
    et_model.fit(X_train, y_train)
    
    y_new_pred = et_model.predict(X_new)
    original_y_new_pred = y_new_pred.copy()


    et_model_all = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.6,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )
    X_all = pd.concat([X_train, X_new])
    y_all = pd.concat([y_train, y_new])

    loo_all = LeaveOneOut()
    loo_scores_all = []
    loo_predictions_all = []
    loo_true_values_all = []

    for train_idx, test_idx in loo_all.split(X_all):
        X_train_fold, X_test_fold = X_all.iloc[train_idx], X_all.iloc[test_idx]
        y_train_fold, y_test_fold = y_all.iloc[train_idx], y_all.iloc[test_idx]
        
        et_model_all.fit(X_train_fold, y_train_fold)
        y_pred_fold = et_model_all.predict(X_test_fold)
        
        mse_fold = mean_squared_error(y_test_fold, y_pred_fold)
        loo_scores_all.append(-mse_fold)
        loo_predictions_all.extend(y_pred_fold)
        loo_true_values_all.extend(y_test_fold)

    from sklearn.base import clone

    et_model_loo = clone(et_model_all)
    et_model_all.fit(X_all, y_all)

    y_train_pred = et_model_all.predict(X_all)

    train_r2 = r2_score(y_all, y_train_pred)
    train_mse = mean_squared_error(y_all, y_train_pred)
    train_mae = mean_absolute_error(y_all, y_train_pred)
    
    loo_r2 = r2_score(loo_true_values_all, loo_predictions_all)
    loo_mse = mean_squared_error(loo_true_values_all, loo_predictions_all)
    loo_mae = mean_absolute_error(loo_true_values_all, loo_predictions_all)
    
    new_r2 = r2_score(y_new, original_y_new_pred)
    new_mse = mean_squared_error(y_new, original_y_new_pred)
    new_mae = mean_absolute_error(y_new, original_y_new_pred)
    
    print("\n📊 RESULTS:")
    print("Metrics on training data:")
    print(f"  R²: {train_r2:.4f}, MSE: {train_mse:.4f}, MAE: {train_mae:.4f}")
    
    print("\nLeave-one-out metrics:")
    print(f"  R²: {loo_r2:.4f}, MSE: {loo_mse:.4f}, MAE: {loo_mae:.4f}")
    print(f"  LOO Score (mean neg_MSE): {np.mean(loo_scores_all):.4f} ± {np.std(loo_scores_all):.4f}")
    
    print("\nMetrics on NEW data (held out from training):")
    print(f"  R²: {new_r2:.4f}, MSE: {new_mse:.4f}, MAE: {new_mae:.4f}")
    
    if picture == True:
        plot_comprehensive_analysis(
            y_all, y_train_pred, 
            loo_true_values_all, loo_predictions_all,
            y_new, original_y_new_pred,
            X_new.index
        )
        
        print_new_predictions_details(X_new, y_new, y_new_pred, X_new.index)
    return {
            'model': et_model_all,
            'LOOCV model': et_model_loo,
            'X_train': X_all,
            'y_train': y_all,
            'X_new': X_new,
            'y_new': y_new,
            'y_new_pred': y_new_pred,
            'loo_scores': loo_scores_all,
            'loo_predictions': loo_predictions_all,
            'loo_true_values': loo_true_values_all,
            'AE_LOOCV': np.abs((np.array(loo_true_values_all) - np.array(loo_predictions_all))),
            'AE_last': np.abs((np.array(y_all) - np.array(y_train_pred))),
            'metrics': {
                'train': {'r2': train_r2, 'mse': train_mse, 'mae': train_mae},
                'loo': {'r2': loo_r2, 'mse': loo_mse, 'mae': loo_mae},
                'new': {'r2': new_r2, 'mse': new_mse, 'mae': new_mae}
            }
    }
        
def plot_comprehensive_analysis(y_train, y_train_pred, loo_true, loo_pred, y_new, y_new_pred, new_indices):
    """
    Комплексная визуализация результатов
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Comprehensive analysis of pretiction \n for "A" using features selected for "B"', fontsize=16, fontweight='bold')
    
    axes[0, 0].scatter(y_train, y_train_pred, alpha=0.6, color='blue', s=50)
    # for i, idx in enumerate(y_train):
    #     axes[0, 0].annotate(f'{i}', (y_train[i], y_train_pred[i]), 
    #                        xytext=(5, 5), textcoords='offset points', fontsize=8)
    min_val = min(y_train.min(), y_train_pred.min())
    max_val = max(y_train.max(), y_train_pred.max())
    axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    axes[0, 0].set_xlabel('True Values')
    axes[0, 0].set_ylabel('Predicted Values')
    axes[0, 0].set_title('Parity Plot: predictions for training on all data after LOOCV')
    axes[0, 0].grid(True, alpha=0.3)
    r2_train = r2_score(y_train, y_train_pred)
    axes[0, 0].text(0.05, 0.95, f'R² = {r2_train:.3f}', transform=axes[0, 0].transAxes,
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    # 2. Parity Plot - LOO
    axes[0, 1].scatter(loo_true, loo_pred, alpha=0.6, color='green', s=50)
    # for i, idx in enumerate(loo_true):
    #     axes[0, 1].annotate(f'{i}', (loo_true[i], loo_pred[i]), 
    #                        xytext=(5, 5), textcoords='offset points', fontsize=8)
    min_val_loo = min(min(loo_true), min(loo_pred))
    max_val_loo = max(max(loo_true), max(loo_pred))
    axes[0, 1].plot([min_val_loo, max_val_loo], [min_val_loo, max_val_loo], 'r--', linewidth=2)
    axes[0, 1].set_xlabel('True Values')
    axes[0, 1].set_ylabel('Predicted Values')
    axes[0, 1].set_title('Parity Plot: predictions for Leave-One-Out cross-validation (LOOCV)')
    axes[0, 1].grid(True, alpha=0.3)
    r2_loo = r2_score(loo_true, loo_pred)
    axes[0, 1].text(0.05, 0.95, f'R² = {r2_loo:.3f}', transform=axes[0, 1].transAxes,
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    axes[0, 2].scatter(y_new, y_new_pred, alpha=0.6, color='red', s=80, edgecolors='black')
    # for i, idx in enumerate(new_indices):
    #     axes[0, 2].annotate(f'{idx}', (y_new.iloc[i], y_new_pred[i]), 
    #                        xytext=(5, 5), textcoords='offset points', fontsize=8)
    min_val_new = min(min(y_new), min(y_new_pred))
    max_val_new = max(max(y_new), max(y_new_pred))
    axes[0, 2].plot([min_val_new, max_val_new], [min_val_new, max_val_new], 'r--', linewidth=2)
    axes[0, 2].set_xlabel('True Values')
    axes[0, 2].set_ylabel('Predicted Values')
    axes[0, 2].set_title('Parity Plot: predictions for hold out sample')
    axes[0, 2].grid(True, alpha=0.3)
    r2_new = r2_score(y_new, y_new_pred)
    axes[0, 2].text(0.05, 0.95, f'R² = {r2_new:.3f}', transform=axes[0, 2].transAxes,
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    residuals_train = y_train - y_train_pred
    axes[1, 0].hist(residuals_train, bins=15, alpha=0.7, color='blue', edgecolor='black')
    axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 0].set_xlabel('Residuals')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of prediction errors \n in training on all data after LOOCV')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].text(0.05, 0.95, f'Mean value: {residuals_train.mean():.3f}\nStandard deviation: {residuals_train.std():.3f}', 
                   transform=axes[1, 0].transAxes, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    residuals_loo = np.array(loo_true) - np.array(loo_pred)
    axes[1, 1].hist(residuals_loo, bins=15, alpha=0.7, color='green', edgecolor='black')
    axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 1].set_xlabel('Residuals')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Residuals distribution errors for \n Leave-One-Out cross-validation (LOOCV)')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].text(0.05, 0.95, f'Mean value: {residuals_loo.mean():.3f}\nStandard deviation: {residuals_loo.std():.3f}', 
                   transform=axes[1, 1].transAxes, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    residuals_new = y_new - y_new_pred
    axes[1, 2].hist(residuals_new, bins=min(10, len(y_new)), alpha=0.7, color='red', edgecolor='black')
    axes[1, 2].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[1, 2].set_xlabel('Residuals')
    axes[1, 2].set_ylabel('Frequency')
    axes[1, 2].set_title('Residuals Distribution errors for hold out sample')
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].text(0.05, 0.95, f'Mean value: {residuals_new.mean():.3f}\nStandard deviation: {residuals_new.std():.3f}', 
                   transform=axes[1, 2].transAxes, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

def print_new_predictions_details(X_new, y_new, y_new_pred, indices):
    """
    Детальная информация о предсказаниях на новых данных
    """
    print("\n" + "=" * 80)
    print("DETAILED PREDICTION REPORT (NEW DATA)")
    print("=" * 80)
    
    predictions_df = pd.DataFrame({
        'Sample index': indices,
        'True value': y_new.values,
        'Predicted value': y_new_pred,
        'Error': y_new.values - y_new_pred,
        'Absolute error': np.abs(y_new.values - y_new_pred),
        'Relative error (%)': np.abs((y_new.values - y_new_pred) / y_new.values) * 100
    })
    
    print(predictions_df.round(4))
    
    print(f"\n📈 СТАТИСТИКА ОШИБОК НА НОВЫХ ДАННЫХ:")
    print(f"  Mean absolute error: {predictions_df['Absolute error'].mean():.4f}")
    print(f"  Max absolute error: {predictions_df['Absolute error'].max():.4f}")
    print(f"  Mean relative error: {predictions_df['Relative error (%)'].mean():.2f}%")
    print(f"  Std of errors: {predictions_df['Error'].std():.4f}")


In [ ]:
print("Running Extra Trees diagnostics...")
results_b = extra_trees_loo_analysis_castom(df_no_zeroB[features_b[:25]], df_no_zeroB["b"], test_size=10, random_state=42, picture=True)

In [ ]:
print("Running Extra Trees diagnostics...")
results_a = extra_trees_loo_analysis_castom(data[features_a[:26]], data["a"], test_size=10, random_state=42, picture=True)

## Inspect LOO outputs
Compare true values, predictions, and error vectors.


In [ ]:
print("Running Extra Trees diagnostics...")
resultsA_b = extra_trees_loo_analysis_castom(data[features_b[:25]], data["a"], test_size=10, random_state=42, picture=True)

In [ ]:
# resultsA_b = extra_trees_loo_analysis_castom(df_no_zeroB[features_a[:25]], df_no_zeroB["b"], test_size=10, random_state=42, picture=True)

In [ ]:
results_b.keys()

In [ ]:
np.abs(np.array(df_no_zeroB["b"]) - np.array(results_b["loo_predictions"])) #- np.array(results_b["AE_LOOCV"])

In [ ]:
np.array(results_b["loo_predictions"])

In [ ]:
np.abs(np.array(data["a"]) - np.array(results_a["loo_true_values"])) 

## Export models
`joblib.dump` trained estimators into the `streamlit/` bundle.


In [ ]:
import joblib

In [ ]:
results_a['model']

In [ ]:
import os
os.getcwd()

In [ ]:
joblib.dump(results_b['model'], 'streamlit/model_b2.pkl')
joblib.dump(features_b[:25], 'streamlit/features_b2.pkl')

In [ ]:
features_a[:26]

In [ ]:
joblib.dump(results_a['model'], 'streamlit/model_a2.pkl')
joblib.dump(features_a[:26], 'streamlit/features_a2.pkl')

## Feature-importance summaries
Quick bar-table helpers for tree importances.


## (continued) Feature importances
Same helper applied to the alternate target pipeline.


In [ ]:
def get_feature_importance_simple(model, feature_names):
    """
    Быстрое получение важности features
    """
    importance = model.feature_importances_
    
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 8))
    top_20 = importance_df.head(20)
    
    plt.barh(top_20['feature'], top_20['importance'], color='lightblue')
    plt.xlabel('Feature importance')
    plt.title('Top 20 feature importances')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Top-10 most important features:")
    print(top_20.head(10).to_string(index=False))
    
    return importance_df

In [ ]:
features_a[:30]

In [ ]:
importance_df = get_feature_importance_simple(results_a["model"], features_a[:26])

In [ ]:
importance_df = get_feature_importance_simple(results_b["model"], features_b[:25])

## SHAP dependency
Install / import SHAP for local explanation plots.


In [ ]:
pip install shap

In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor

In [ ]:
def shap_analysis_extra_trees(model, X, y=None, data = None, sample_size=100, random_state=42, rows_to_annotate = None, threshold = 0):
    """
    SHAP analysis for Extra Trees regression
    """
    print("=" * 80)
    print("SHAP ANALYSIS FOR EXTRA TREES REGRESSION")
    print("=" * 80)
    
    if len(X) > sample_size:
        X_sample = X.sample(n=sample_size, random_state=random_state)
        print(f"Using sample of {sample_size} instances for SHAP analysis")
    else:
        X_sample = X
        print(f"Using all {len(X)} instances for SHAP analysis")
    
    explainer = shap.TreeExplainer(model)
    
    print("Calculating SHAP values...")
    shap_values = explainer.shap_values(X_sample)
    
    expected_value = explainer.expected_value
    if isinstance(expected_value, np.ndarray):
        expected_value = expected_value[0] if len(expected_value) > 0 else expected_value
    # print(f"Base value (expected value): {expected_value:.4f}")
    print(f"SHAP values shape: {shap_values.shape}")
    
    shap_explanation = shap.Explanation(
        values=shap_values,
        base_values=expected_value,
        data=X_sample.values,
        feature_names=X_sample.columns.tolist()
    )
    
    # print("\n📈 Creating summary plot...")
    # plt.figure(figsize=(12, 8))
    # shap.summary_plot(shap_values, X_sample, show=False)
    # plt.title("SHAP Summary Plot - Feature Importance", fontsize=16, fontweight='bold')
    # plt.tight_layout()
    # plt.show()

    print("\n📈 Creating summary plot with index annotations...")
    plt.figure(figsize=(12, 8))
    
    shap.summary_plot(shap_values, X_sample, show=False)
    
    shap_df = _add_simple_index_annotations(shap_values, X_sample, df = data, threshold=threshold, rows_to_annotate= rows_to_annotate)
    
    plt.title(f"SHAP Summary Plot - Feature Importance)", 
              fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("📊 Creating feature importance plot...")
    plt.figure(figsize=(12, 6))
    shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
    plt.title("SHAP Feature Importance (Mean |SHAP|)", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # 3. Beeswarm plot
    print("🐝 Creating beeswarm plot...")
    plt.figure(figsize=(12, 8))
    shap.plots.beeswarm(shap_explanation, show=False)
    plt.title("SHAP Beeswarm Plot - Feature Effects", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("🌊 Creating waterfall plot for first instance...")
    plt.figure(figsize=(14, 8))
    shap.plots.waterfall(shap_explanation[0], show=False, max_display=15)
    plt.title(f"SHAP Waterfall Plot - Instance: {X_sample.index[0]}", 
              fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("⚡ Creating force plot...")
    plt.figure(figsize=(14, 4))
    shap.plots.force(shap_explanation[0], matplotlib=True, show=False)
    plt.title(f"SHAP Force Plot - Instance: {X_sample.index[0]}", 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("📈 Creating decision plot...")
    plt.figure(figsize=(14, 10))
    shap.decision_plot(expected_value, shap_values[:10], 
                      X_sample.iloc[:10], 
                      feature_names=list(X_sample.columns),
                      show=False)
    plt.title("SHAP Decision Plot (First 10 Instances)", 
              fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("📊 Creating dependence plots for top 4 features...")
    top_features = get_top_features_from_shap(shap_values, X_sample.columns, n=4)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.ravel()
    
    for i, feature in enumerate(top_features):
        shap.dependence_plot(feature, shap_values, X_sample, 
                           ax=axes[i], show=False)
        axes[i].set_title(f'SHAP Dependence: {feature}', fontweight='bold', fontsize=12)
        axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    importance_df = get_shap_feature_importance(shap_values, X_sample.columns)
    
    results = {
        'explainer': explainer,
        'shap_values': shap_values,
        'expected_value': expected_value,
        'X_sample': X_sample,
        'feature_importance': importance_df,
        'shap_explanation': shap_explanation
    }
    
    return results, shap_df

def _add_simple_index_annotations(shap_values, X_sample, df = None, threshold=0, rows_to_annotate = None):
    """
    Версия с умным размещением по зонам и выводом данных
    """
    ax = plt.gca()
    y_tick_labels = [label.get_text() for label in ax.get_yticklabels()]
    feature_names = X_sample.columns.tolist()
    
    position_to_feature_idx = {}
    for position, feature_name in enumerate(y_tick_labels):
        if feature_name in feature_names and feature_name in rows_to_annotate:
            print(feature_name)
            position_to_feature_idx[position] = feature_names.index(feature_name)
    
    annotation_zones = {}
    
    outlier_data = []
    
    for position, feature_idx in position_to_feature_idx.items():
        feature_name = feature_names[feature_idx]
        feature_shap = shap_values[:, feature_idx]
        high_impact_indices = np.where(np.abs(feature_shap) > threshold)[0]
        
        x_positions = []
        for idx in high_impact_indices:
            shap_val = feature_shap[idx]
            sample_index = X_sample.index[idx]
            feature_value = X_sample.iloc[idx, feature_idx]
            
            outlier_data.append({
                'feature': feature_name,
                'feature_value': feature_value,
                'index': sample_index,
                'reaction_name': df["inp+inp=out"][sample_index],
                'shap_value': shap_val,
                'abs_shap': abs(shap_val)
            })
            
            x_positions.append((shap_val, sample_index, position))
        
        x_positions.sort(key=lambda x: x[0])
        
        for i, (x, text, y_pos) in enumerate(x_positions):
            y_offset = 10 if i % 2 == 0 else -10
            x_offset = 5 + (i % 3) * 8
            
            ax.annotate(text,
                       xy=(x, y_pos),
                       xytext=(x_offset, y_offset),
                       textcoords='offset points',
                       fontsize=6,
                       alpha=0.7,
                       bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.5))
    
    _print_outlier_table_pandas(outlier_data, threshold)
    
    return outlier_data

def _print_outlier_table_pandas(outlier_data, threshold):
    """
    Выводит таблицу с использованием pandas для красивого форматирования
    """
    if not outlier_data:
        print(f"\n📊 Нет точек с |SHAP| > {threshold:.2f}")
        return
    
    import pandas as pd
    
    df = pd.DataFrame(outlier_data)
    df = df.sort_values('feature', ascending=False)

    # global aut_SHAP_b
    # aut_SHAP_b = df
    
    # df_display = df[['feature', 'shap_value', 'index', 'feature_value', 'abs_shap']].copy()
    
    print(f"\n📊 ТОЧКИ С ВЫСОКИМ ВЛИЯНИЕМ (|SHAP| > {threshold:.2f})")
    print("=" * 80)
    
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 20)

    print(df.to_string(index=False, formatters={
        'SHAP value': '{:>10.3f}'.format,
        'Feature value': '{:>10.3f}'.format,
        '|SHAP|': '{:>8.3f}'.format
    }))
    
    print(f"\nВсего точек: {len(df)}")
    print(f"Диапазон |SHAP|: {df['abs_shap'].min():.3f} - {df['abs_shap'].max():.3f}")
    

def _print_outlier_table(outlier_data, threshold):
    """
    Выводит красивую таблицу с информацией о выбивающихся точках
    """
    if not outlier_data:
        print(f"\n📊 Нет точек с |SHAP| > {threshold:.2f}")
        return
    
    print(f"\n📊 ТОЧКИ С ВЫСОКИМ ВЛИЯНИЕМ (|SHAP| > {threshold:.2f})")
    print("=" * 85)
    
    outlier_data.sort(key=lambda x: x['abs_shap'], reverse=True)
    
    print(f"{'Фича':<25} {'SHAP value':<15} {'Индекс':<12} {'Feature value':<15} {'|SHAP|':<10}")
    print("-" * 85)
    
    for data in outlier_data:
        print(f"{data['feature']:<25} {data['shap_value']:>12.3f}  {data['index']:<12} {data['feature_value']:>12.3f}  {data['abs_shap']:>8.3f}")
    
    print("-" * 85)
    print(f"Всего точек: {len(outlier_data)}")
    
    if outlier_data:
        max_shap = max(outlier_data, key=lambda x: x['abs_shap'])
        min_shap = min(outlier_data, key=lambda x: x['abs_shap'])
        print(f"Максимальное |SHAP|: {max_shap['abs_shap']:.3f} (фича: {max_shap['feature']}, индекс: {max_shap['index']})")
        print(f"Минимальное |SHAP|: {min_shap['abs_shap']:.3f} (фича: {min_shap['feature']}, индекс: {min_shap['index']})")
        
def get_shap_feature_importance(shap_values, feature_names):
    """
    Вычисление важности features на основе SHAP
    """
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'shap_importance': np.abs(shap_values).mean(axis=0),
        'shap_std': np.abs(shap_values).std(axis=0)
    }).sort_values('shap_importance', ascending=False)
    
    print("\n🎯 Top 15 Features by SHAP Importance:")
    print(importance_df.head(15).to_string(index=False, float_format='%.4f'))
    
    return importance_df

def get_top_features_from_shap(shap_values, feature_names, n=10):
    """
    Получить топ-N самых важных features по SHAP
    """
    importance = np.abs(shap_values).mean(axis=0)
    top_indices = np.argsort(importance)[::-1][:n]
    return [feature_names[i] for i in top_indices]

In [ ]:
results_a.keys()

In [ ]:
# aut_SHAP_b = None
_, shap_resultsB = shap_analysis_extra_trees(
    model=results_b["model"], 
    X = df_no_zeroB[features_b[:25]], 
    data = df_no_zeroB,
    y = df_no_zeroB["b"],
    rows_to_annotate = ["reagent2_GATS1p", "reagent2_AATS1p", "reagent2_AATSC3p", "reagent2_TPSA", "reagent2_RPSA"],
    threshold = 2.5

    )
df_shap_b = pd.DataFrame(shap_resultsB)
df_shap_b = df_shap_b.sort_values('feature', ascending=False)

In [ ]:
data.head(10)

In [ ]:
df_shap_b.head(10)

In [ ]:
df_shap_b= df_shap_b.reset_index()
df_shap_b = df_shap_b[df_shap_b.columns[1:]]

In [ ]:
df_shap_b.head(6)

In [ ]:
interected_d = ["reagent2_GATS1p", "reagent2_AATS1p", "reagent2_AATSC3p", "reagent2_TPSA", "reagent2_RPSA"]

In [ ]:
selected_cities_b = df_shap_b[df_shap_b['feature'].isin(interected_d)]
selected = df_shap_b[(df_shap_b['feature'] == "reagent2_RPSA") & (df_shap_b['shap_value'] < 5)]

In [ ]:
selected

In [ ]:
# df_no_zeroB = df_no_zeroB[df_no_zeroB.columns[2:]]

In [ ]:
# df_no_zeroB.to_csv('../data/df_no_zeroB.csv')

In [ ]:
# data = data[data.columns[1:]]

In [ ]:
# data.to_csv('../data/data_a.csv')

## SHAP for regime **A**
Subset SHAP tables for hand-picked descriptors of interest.


In [ ]:
interested_a = ["reagent2_MATS1se", "reagent1_C2SP2"]
_, shap_resultsA = shap_analysis_extra_trees(
    model=results_a["model"],
    X = data[features_a[:26]], 
    y = data["a"], 
    data = data,
    rows_to_annotate = interested_a,
    threshold = 2.5
    )
df_shap_a = pd.DataFrame(shap_resultsA)
df_shap_a = df_shap_a.sort_values('feature', ascending=False)

In [ ]:
df_shap_a = df_shap_a.reset_index()


In [ ]:
df_shap_a = df_shap_a[df_shap_a.columns[1:]]

In [ ]:
df_shap_a.head(10)

In [ ]:
interected_a = ["reagent2_MATS1se", "reagent1_C2SP2"]

In [ ]:
selected_cities_a = df_shap_a[df_shap_a['feature'].isin(interected_d)]
selected = df_shap_a[(df_shap_a['feature'] == "reagent1_C2SP2") & (df_shap_a['shap_value'] < -2.5)]

In [ ]:
selected

In [ ]:
selected.sort_values('shap_value', ascending=False)

## Combined SHAP view
Merge descriptor sets from both regimes for a joint explanation pass.


In [ ]:
interected_ba = interected_a + interected_d

In [ ]:
_, shap_resultsA_b = shap_analysis_extra_trees(
    model=resultsA_b["model"],
    X = data[features_b[:25]], 
    y = data["a"], 
    data = data,
    rows_to_annotate = interected_a + interected_d,
    threshold=5
    )
df_shap_ab = pd.DataFrame(shap_resultsA_b)
df_shap_ab = df_shap_ab.sort_values('feature', ascending=False)

In [ ]:
df_shap_ab = df_shap_ab.reset_index()

In [ ]:
df_shap_ab = df_shap_ab[df_shap_ab.columns[1:]]

In [ ]:
df_shap_ab.head(5)

In [ ]:
interected_ba

In [ ]:
selected_cities_b = df_shap_ab[df_shap_ab['feature'].isin(interected_ba)]
selected = df_shap_ab[(df_shap_ab['feature'] == "reagent2_GATS1p") & (df_shap_ab['shap_value'] < -7)]

In [ ]:
selected

In [ ]:
print("Prediction matrix shape:", data[features_a[:26]].shape)
print("Training matrix shape:", results_a["X_train"].shape)

In [ ]:
print("Prediction matrix shape:", df_no_zeroB[features_b[:25]].shape)
print("Training matrix shape:", results_b["X_train"].shape)

In [ ]:
def check_feature_order(model, data, features):
    """Checks consistency of feature order between model and data."""
    
    print("=" * 60)
    print("FEATURE ORDER CHECK")
    print("=" * 60)
    
    if hasattr(model, 'feature_names_in_'):
        model_features = list(model.feature_names_in_)
        print(f"✅ Модель хранит имена features ({len(model_features)} шт.)")
        print("First five model features:", model_features[:5])
    else:
        print("❌ Model does NOT store feature names")
        model_features = None
    
    data_features = list(data[features].columns)
    print(f"Признаки в данных ({len(data_features)} шт.):", data_features[:5])
    
    if model_features and data_features:
        if model_features == data_features:
            print("✅ Feature order MATCHES")
        else:
            print("❌ Feature order DIFFERS!")
            print("First five feature-order differences:")
            for i in range(min(5, len(model_features), len(data_features))):
                status = "==" if model_features[i] == data_features[i] else "!="
                print(f"  Position {i}: model '{model_features[i]}' {status} данные '{data_features[i]}'")
    
    return model_features, data_features

model_features, data_features = check_feature_order(
    results_b["model"], df_no_zeroB, features_b[:25]
)

In [ ]:
def safe_predict(model, data, features, model_name=""):
    """
    Безопасное предсказание с проверкой порядка features
    """
    print(f"🔍 Предсказание для {model_name}")
    
    if hasattr(model, 'feature_names_in_'):
        expected_features = list(model.feature_names_in_)
    else:
        expected_features = features
        print("⚠️ Model has no feature names; using supplied list")
    
    print(f"Модель ожидает {len(expected_features)} features")
    
    missing_features = [f for f in expected_features if f not in data.columns]
    if missing_features:
        print(f"❌ Отсутствуют featureи: {missing_features}")
        return None
    
    ordered_data = data[expected_features]
    
    print(f"✅ Данные подготовлены: {ordered_data.shape}")
    print("First five columns in model order:", list(ordered_data.columns[:5]))
    
    predictions = model.predict(ordered_data)
    
    return predictions

In [ ]:
def test_feature_order_issue():
    """Tests feature-order alignment issues."""
    
    results = extra_trees_loo_analysis_castom(
        data[features_a[:26]], data["a"], test_size=10, random_state=42
    )
    
    correct_predictions = safe_predict(
        results["model"], data[features_a[:26]], features_a[:26]
    )
    
    shuffled_features = features_a[:26].copy()
    random.shuffle(shuffled_features)
    
    wrong_predictions = safe_predict(
        results["model"], data[shuffled_features], shuffled_features
    )
    
    if correct_predictions is not None and wrong_predictions is not None:
        differences = abs(correct_predictions - wrong_predictions)
        print(f"📊 Максимальная разница из-за порядка: {differences.max():.6f}")
        print(f"📊 Средняя разница: {differences.mean():.6f}")
        
        if differences.max() > 0.001:
            print("❌ Feature order AFFECTS predictions!")

test_feature_order_issue()